OBJECTIVE: Create a restaurant recommendation
system based on user preferences.

A Restaurant recommendation
system based on cuisines

In [1]:
import pandas as pd
import numpy as np
import warnings
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [3]:
Data=pd.read_csv("Cognifyz_Restaurant_Dataset.csv",skipinitialspace=True)

In [4]:
Data.shape

(9551, 21)

In [5]:
Data.head(2)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591


In [6]:
Data['Address'][1]

'Little Tokyo, 2277 Chino Roces Avenue, Legaspi Village, Makati City'

In [7]:
Data.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [8]:
Data[Data['Price range']==1].head(2)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
23,6601005,Caf�� Daniel Briand,30,Bras�_lia,"SCLN 104, Bloco A, Loja 26, Asa Norte, Bras�_lia",Asa Norte,"Asa Norte, Bras�_lia",-47.882667,-15.7775,Cafe,...,Brazilian Real(R$),No,No,No,No,1,3.8,Yellow,Good,9
30,6600060,Sandubas Caf��,30,Bras�_lia,"Edif�_cio Jos�� Severo, SCS 6, Bloco A, Loja 9...",Asa Sul,"Asa Sul, Bras�_lia",-47.890167,-15.7970,"Brazilian, Cafe",...,Brazilian Real(R$),No,No,No,No,1,0.0,White,Not rated,2


In [9]:
Data[Data['Price range']==4].head(2)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365


                  Price Range interpretations
    1 -   Cheap
    2 -   Affordable
    3 -   Expensive
    4 -   Very Expensive


In [10]:
Data['Cuisines'].dtypes

dtype('O')

In [11]:
# importing the required libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from scipy.spatial.distance import cdist

In [12]:
# converting criterias[text dtypes] to vector
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
Data['Cuisines']=Data['Cuisines'].fillna(" ")
tfidf_matrix=tfidf_vectorizer.fit_transform(Data['Cuisines'])

# text frequencies and their significance into vector

In [13]:
tfidf_matrix.shape

(9551, 148)

In [14]:
# computing cosine similarities to get the frequency significances/relationship and references of restaurants
cos_sim=linear_kernel(tfidf_matrix,tfidf_matrix)

In [15]:
cos_sim

array([[1.        , 0.56345605, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.56345605, 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.3880504 ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.3880504 ,
        1.        ]], shape=(9551, 9551))

In [16]:
Data['Restaurant Name'].duplicated().sum()

np.int64(2105)

In [17]:
# getting the indices:name of the restaurant and respective indexes
indices=pd.Series(Data.index,index=Data['Restaurant Name']).drop_duplicates()

In [18]:
indices

Restaurant Name
Le Petit Souffle               0
Izakaya Kikufuji               1
Heat - Edsa Shangri-La         2
Ooma                           3
Sambo Kojin                    4
                            ... 
Naml۱ Gurme                 9546
Ceviz A��ac۱                9547
Huqqa                       9548
A���k Kahve                 9549
Walter's Coffee Roastery    9550
Length: 9551, dtype: int64

In [19]:
indices['Ooma']

np.int64(3)

In [20]:
# Creating the restaurant recommendation function

def get_recommendation(name, cos_sim =cos_sim):
    idx=indices[name]
    sim_score=enumerate(cos_sim[idx]) 
    
    # sorting the similarities and reverse it 
    sim_score=sorted(sim_score, key=lambda x: x[1], reverse=True)

    # selecting top 10 Restaurant
    sim_score=sim_score[:10]

    # goes back to first index of similarity score
    sim_index= [i[0] for i in sim_score]

    print(Data['Restaurant Name'].iloc[sim_index])
        

In [21]:
get_recommendation('Huqqa')

9548                       Huqqa
9524                Gaga Manjero
9525                     Cafemiz
9547                Ceviz A��ac۱
2560                  Food Cloud
31                  Villa Tevere
40                          Gero
53              D.O.C Ristorante
76              Terra�_o It��lia
115     La Dolce Vita Ristorante
Name: Restaurant Name, dtype: object


In [22]:
get_recommendation('Ooma')

3                              Ooma
71                       Kawa Sushi
185                     Tokyo Sushi
215        Fuji Japanese Steakhouse
247                           Osaka
260                        Miyabi 9
383                         Ichiban
480    Fuji Bay Japanese Restaurant
531                 Masato Japanese
549                          Sakura
Name: Restaurant Name, dtype: object


In [23]:
# testing
get_recommendation('Huqqa')

9548                       Huqqa
9524                Gaga Manjero
9525                     Cafemiz
9547                Ceviz A��ac۱
2560                  Food Cloud
31                  Villa Tevere
40                          Gero
53              D.O.C Ristorante
76              Terra�_o It��lia
115     La Dolce Vita Ristorante
Name: Restaurant Name, dtype: object
